# SpaCy Span Categorization Model Training

This notebook demonstrates the end-to-end process of training a SpaCy span categorization (spancat) model for entity extraction. The process includes:

1. Data preparation
2. Model training
3. Model evaluation
4. Inference on new documents

## Setup and Requirements

Required packages:
- spacy
- pandas
- numpy
- scikit-learn (for evaluation metrics)

In [ ]:
import spacy
import pandas as pd
import numpy as np
from pathlib import Path
from spacy.tokens import DocBin
from sklearn.metrics import classification_report
import json
import random

## 1. Data Preparation

We'll create functions to load and prepare our training data. The input will be:
- Raw text files
- Entity annotations in the format: {text: str, entities: List[{start: int, end: int, label: str}]}

In [ ]:
def load_data(text_dir: Path, entities_dir: Path):
    """Load raw texts and their corresponding entity annotations."""
    data = []
    
    # Iterate through text files
    for text_file in text_dir.glob("*.txt"):
        # Find corresponding entity file
        entity_file = entities_dir / f"{text_file.stem}.json"
        
        if entity_file.exists():
            # Load text
            with open(text_file, 'r', encoding='utf-8') as f:
                text = f.read()
            
            # Load entities
            with open(entity_file, 'r', encoding='utf-8') as f:
                entities = json.load(f)
            
            data.append({
                'text': text,
                'entities': entities
            })
    
    return data

def prepare_training_data(data, nlp):
    """Convert raw data into spaCy's DocBin format for training."""
    db = DocBin()
    
    for example in data:
        text = example['text']
        doc = nlp.make_doc(text)
        
        # Convert entity annotations to spans
        spans = []
        for ent in example['entities']:
            span = doc.char_span(
                ent['start'],
                ent['end'],
                label=ent['label']
            )
            if span is not None:  # Only add valid spans
                spans.append(span)
        
        doc.spans["sc"] = spans  # 'sc' is the key for span categorizer
        db.add(doc)
    
    return db

## 2. Model Configuration and Training

We'll set up a basic spancat pipeline and train it on our data.

In [ ]:
def create_spancat_config(labels):
    """Create a basic spancat configuration."""
    config = {
        "threshold": 0.5,
        "spans_key": "sc",
        "max_positive": 1,
        "scorer": {"@scorers": "spacy.spancat_scorer.v1"},
        "spans_key": "sc",
        "positive_label": True,
        "candidates_key": "candidates"
    }
    return config

def train_spancat_model(train_data, val_data=None, n_iter=10):
    """Train a spancat model from scratch."""
    # Create blank model
    nlp = spacy.blank("en")
    
    # Add spancat component
    spancat = nlp.add_pipe("spancat", config=create_spancat_config([]))
    
    # Prepare training data
    train_docs = list(train_data.get_docs(nlp.vocab))
    if val_data:
        val_docs = list(val_data.get_docs(nlp.vocab))
    else:
        val_docs = None
    
    # Train the model
    losses = {}
    optimizer = nlp.initialize()
    for i in range(n_iter):
        random.shuffle(train_docs)
        for doc in train_docs:
            example = {"doc": doc}
            nlp.update([example], sgd=optimizer, losses=losses)
        print(f"Iteration {i+1}, Losses:", losses)
    
    return nlp

## 3. Model Evaluation

We'll create functions to evaluate the model's performance.

In [ ]:
def evaluate_model(nlp, test_data):
    """Evaluate model performance on test data."""
    y_true = []
    y_pred = []
    
    for doc in test_data.get_docs(nlp.vocab):
        # Get true spans
        true_spans = {(span.start, span.end, span.label_) for span in doc.spans["sc"]}
        
        # Get predicted spans
        pred_doc = nlp(doc.text)
        pred_spans = {(span.start, span.end, span.label_) for span in pred_doc.spans["sc"]}
        
        # Add to lists
        y_true.extend([1 if span in true_spans else 0 for span in pred_spans])
        y_pred.extend([1 if span in pred_spans else 0 for span in true_spans])
    
    # Calculate metrics
    report = classification_report(y_true, y_pred)
    return report

## 4. Inference

Finally, we'll create a function to use the trained model for inference on new documents.

In [ ]:
def predict_entities(nlp, text):
    """Use the trained model to predict entities in new text."""
    doc = nlp(text)
    predictions = []
    
    for span in doc.spans["sc"]:
        predictions.append({
            'text': span.text,
            'start': span.start_char,
            'end': span.end_char,
            'label': span.label_
        })
    
    return predictions

## Example Usage

Here's how to use the functions above for a complete training and inference pipeline:

In [ ]:
# 1. Load and prepare data
text_dir = Path('../data/raw')
entities_dir = Path('../data/entities')

data = load_data(text_dir, entities_dir)

# Split data into train/test
random.shuffle(data)
split = int(len(data) * 0.8)
train_data = data[:split]
test_data = data[split:]

# Create blank model and prepare DocBins
nlp = spacy.blank("en")
train_db = prepare_training_data(train_data, nlp)
test_db = prepare_training_data(test_data, nlp)

# 2. Train model
trained_nlp = train_spancat_model(train_db, n_iter=10)

# 3. Evaluate model
evaluation_report = evaluate_model(trained_nlp, test_db)
print("\nEvaluation Report:")
print(evaluation_report)

# 4. Save model
trained_nlp.to_disk("../models/spancat_model")

# 5. Example inference
sample_text = "Your sample text here"
predictions = predict_entities(trained_nlp, sample_text)
print("\nPredicted Entities:")
print(json.dumps(predictions, indent=2))